# Task 4 — Luminance robustness

Test system effectiveness when the **Y channel** (YCbCr) is modified in three ways:

| Transform | Variants |
|---|---|
| Quadratic | Y_new = (Y/255)² × 255 |
| Linear scaling | × 1/2, × 3/5, × 3/4, × 4/3, × 3/2 |
| Constant offset | −100, −20, −10, +30 |

All values are clamped to [0, 255] before converting back to BGR.  
Each experiment tests genuine pairs **and** impostor pairs from `data/enrolled_test/`.

## 1. Setup

In [ ]:
import sys
import random
import warnings
from pathlib import Path
from fractions import Fraction

import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path('..').resolve()))

from src.model import get_insightface_model, get_embedding
from src.database import EmbeddingDB
from src.degradation import (
    apply_luminance_quadratic,
    apply_luminance_linear,  LUMINANCE_LINEAR_COEFFS,
    apply_luminance_offset,  LUMINANCE_OFFSETS,
)
from src.metrics import compute_far_frr, compute_roc, eer
from src.utils import list_images

warnings.filterwarnings('ignore')

SEED           = 44
DEFAULT_THRESH = 0.4
TEST_DIR       = Path('../data/enrolled_test')

random.seed(SEED)
np.random.seed(SEED)

print('Loading model...')
app = get_insightface_model('buffalo_l', ctx_id=0)
db  = EmbeddingDB.from_file()

enrolled_users = set(db.get_all_users())
user_list      = sorted(enrolled_users)
print(f'Enrolled users : {len(enrolled_users)}')

## 2. Load enrolled test images + precompute clean embeddings

In [ ]:
print('Loading enrolled_test images ...')

all_probes = []   # (user_id, img_bgr)
for user_dir in sorted(d for d in TEST_DIR.iterdir()
                       if d.is_dir() and d.name in enrolled_users):
    for img_path in list_images(user_dir):
        img = cv2.imread(str(img_path))
        if img is not None:
            all_probes.append((user_dir.name, img))

print(f'Total probes : {len(all_probes)}')

## 3. Visualise luminance transforms

In [ ]:
sample_img = all_probes[0][1]

transforms_vis = (
    [('Original',    sample_img)] +
    [('Quadratic',   apply_luminance_quadratic(sample_img))] +
    [(f'×{Fraction(c).limit_denominator(10)}', apply_luminance_linear(sample_img, c))
     for c in LUMINANCE_LINEAR_COEFFS] +
    [(f'{o:+d}',     apply_luminance_offset(sample_img, o))
     for o in LUMINANCE_OFFSETS]
)

n = len(transforms_vis)
fig, axes = plt.subplots(1, n, figsize=(2.5 * n, 3))
for ax, (title, img) in zip(axes, transforms_vis):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(title, fontsize=9)
    ax.axis('off')

plt.suptitle('Luminance transforms applied to sample image', y=1.02)
plt.tight_layout()

out_dir = Path('../results/task4')
out_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(out_dir / 'transform_examples.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Experiment runner

In [ ]:
rng = random.Random(SEED)

def run_experiment(transform_fn, label: str) -> dict:
    """
    Apply transform_fn to each probe image, score against genuine and impostor
    enrolled embeddings, return metrics.
    """
    scores_gen = []
    scores_imp = []

    for user_id, img in all_probes:
        t_img = transform_fn(img)
        emb   = get_embedding(app, t_img, fallback=True)
        if emb is None:
            continue

        # Genuine
        refs = db.get_user_embeddings(user_id)
        if refs:
            scores_gen.append(max(float(np.dot(emb, r)) for r in refs))

        # Impostor
        candidates = [u for u in user_list if u != user_id]
        if candidates:
            imp_refs = db.get_user_embeddings(rng.choice(candidates))
            if imp_refs:
                scores_imp.append(max(float(np.dot(emb, r)) for r in imp_refs))

    scores = np.concatenate([scores_gen, scores_imp])
    labels = np.concatenate([np.ones(len(scores_gen)), np.zeros(len(scores_imp))])

    far, frr, acc       = compute_far_frr(scores, labels, DEFAULT_THRESH)
    fpr, tpr, thr       = compute_roc(scores, labels)
    eer_val, _          = eer(fpr, tpr, thr)
    auc_val             = float(np.trapz(tpr, fpr))

    return {
        'label': label,
        'scores_gen': np.array(scores_gen),
        'scores_imp': np.array(scores_imp),
        'scores': scores, 'labels': labels,
        'FAR': far, 'FRR': frr, 'Accuracy': acc,
        'EER': eer_val, 'AUC': auc_val,
        'n_genuine': len(scores_gen), 'n_impostor': len(scores_imp),
        'fpr': fpr, 'tpr': tpr,
    }

## 5. Run all experiments

In [ ]:
results = []

# ── Quadratic ─────────────────────────────────────────────────────────────────
print('Quadratic ...')
results.append(run_experiment(apply_luminance_quadratic, 'Quadratic'))

# ── Linear ────────────────────────────────────────────────────────────────────
coeff_labels = ['×1/2', '×3/5', '×3/4', '×4/3', '×3/2']
for coeff, label in zip(LUMINANCE_LINEAR_COEFFS, coeff_labels):
    print(f'Linear {label} ...')
    results.append(run_experiment(
        lambda img, c=coeff: apply_luminance_linear(img, c),
        f'Linear {label}'
    ))

# ── Offset ────────────────────────────────────────────────────────────────────
for offset in LUMINANCE_OFFSETS:
    label = f'Offset {offset:+d}'
    print(f'{label} ...')
    results.append(run_experiment(
        lambda img, o=offset: apply_luminance_offset(img, o),
        label
    ))

print(f'\nDone — {len(results)} experiments.')

## 6. Score distributions

In [ ]:
n_cols = 5
n_rows = (len(results) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows), sharey=True)
axes_flat = axes.flatten()

for ax, res in zip(axes_flat, results):
    ax.hist(res['scores_gen'], bins=35, alpha=0.65, color='steelblue', label='Genuine')
    ax.hist(res['scores_imp'], bins=35, alpha=0.65, color='tomato',    label='Impostor')
    ax.axvline(DEFAULT_THRESH, color='k', ls='--', lw=1.0)
    ax.set_title(res['label'], fontsize=9)
    ax.set_xlabel('Cosine sim', fontsize=8)

for ax in axes_flat[len(results):]:
    ax.set_visible(False)

axes_flat[0].legend(fontsize=7)
plt.suptitle('Score distributions per luminance transform', y=1.01)
plt.tight_layout()
plt.savefig(out_dir / 'score_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Metrics comparison

In [ ]:
labels_list = [r['label']    for r in results]
fars        = [r['FAR']      for r in results]
frrs        = [r['FRR']      for r in results]
accs        = [r['Accuracy'] for r in results]
eers        = [r['EER']      for r in results]

x = np.arange(len(results))
width = 0.2

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Bar chart
axes[0].bar(x - width,     [f*100 for f in fars],  width, label='FAR',      color='tomato')
axes[0].bar(x,             [f*100 for f in frrs],  width, label='FRR',      color='steelblue')
axes[0].bar(x + width,     [e*100 for e in eers],  width, label='EER',      color='orange')
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels_list, rotation=40, ha='right', fontsize=8)
axes[0].set_ylabel('%')
axes[0].set_title('FAR / FRR / EER per luminance transform')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Accuracy line
axes[1].plot(x, [a*100 for a in accs], 'o-', color='seagreen', lw=2)
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels_list, rotation=40, ha='right', fontsize=8)
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Accuracy per luminance transform')
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 105)

# Shade groups
for ax in axes:
    ax.axvspan(-0.5, 0.5, alpha=0.06, color='purple', label='Quadratic')
    ax.axvspan(0.5,  5.5, alpha=0.06, color='blue',   label='Linear')
    ax.axvspan(5.5,  9.5, alpha=0.06, color='green',  label='Offset')

plt.tight_layout()
plt.savefig(out_dir / 'metrics_comparison.png', dpi=150)
plt.show()

## 8. ROC curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

groups = [
    ('Quadratic',  results[:1],  plt.cm.Purples),
    ('Linear',     results[1:6], plt.cm.Blues),
    ('Offset',     results[6:],  plt.cm.Greens),
]

for ax, (group_name, group_results, cmap) in zip(axes, groups):
    colors = cmap(np.linspace(0.4, 0.9, max(len(group_results), 1)))
    for res, color in zip(group_results, colors):
        ax.plot(res['fpr'], res['tpr'], color=color,
                label=f"{res['label']}  AUC={res['AUC']:.3f}")
    ax.plot([0, 1], [0, 1], 'k--', lw=0.8)
    ax.set_xlabel('FAR')
    ax.set_ylabel('1 - FRR')
    ax.set_title(f'ROC — {group_name}')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(out_dir / 'roc_curves.png', dpi=150)
plt.show()

## 9. Summary table

In [ ]:
rows = []
for res in results:
    rows.append({
        'Transform':    res['label'],
        'Genuine':      res['n_genuine'],
        'Impostor':     res['n_impostor'],
        'FAR (%)':      round(res['FAR']      * 100, 2),
        'FRR (%)':      round(res['FRR']      * 100, 2),
        'Accuracy (%)': round(res['Accuracy'] * 100, 2),
        'EER (%)':      round(res['EER']      * 100, 2),
        'AUC':          round(res['AUC'],            4),
    })

df = pd.DataFrame(rows)
df_styled = (
    df.style
      .highlight_max(subset=['FAR (%)'],      color='#f4cccc')
      .highlight_max(subset=['FRR (%)'],      color='#fce5cd')
      .highlight_max(subset=['Accuracy (%)'], color='#d9ead3')
      .highlight_min(subset=['EER (%)'],      color='#d9ead3')
      .highlight_min(subset=['AUC'],          color='#f4cccc')
)
display(df_styled)

df.to_csv(out_dir / 'luminance_results.csv', index=False)
print(f'Saved → {out_dir / "luminance_results.csv"}')

## 10. Summary

In [ ]:
print('=' * 72)
print('TASK 4 — LUMINANCE ROBUSTNESS RESULTS')
print(f'Threshold : {DEFAULT_THRESH}')
print('=' * 72)
print(f'{"Transform":<18} {"n":>6} {"FAR":>8} {"FRR":>8} {"Acc":>8} {"EER":>8} {"AUC":>7}')
print('-' * 72)
for res in results:
    n = res['n_genuine'] + res['n_impostor']
    print(f"{res['label']:<18} {n:>6}  "
          f"{res['FAR']*100:>7.2f}%  {res['FRR']*100:>7.2f}%  "
          f"{res['Accuracy']*100:>7.2f}%  {res['EER']*100:>7.2f}%  {res['AUC']:>6.4f}")
print('=' * 72)